# Praktikum 2 - Deployment ke Hugging Face (Docker + Flask)

**Nama:** Radit Firansah  
**NIM:** 244107020196

Menyiapkan aplikasi Flask, requirements, dan Dockerfile, lalu menguji inferensi model secara lokal.

## Deskripsi

Pada praktikum ini kita melakukan siklus MLOps sederhana: model Day vs Night hasil Praktikum 1 di-deploy ke Hugging Face Spaces menggunakan Docker + Flask.

**Langkah deployment di Hugging Face (ringkas):**

1. **Persiapan Environment** — buat folder proyek, siapkan `day_night_model.h5` dan `scaler.pkl`.
2. **Konfigurasi Server (`app.py`)** — aplikasi Flask yang memuat model + scaler dan melakukan preprocessing HOG yang sama persis dengan training.
3. **Konfigurasi Dependencies (`requirements.txt`)** — daftar library yang dibutuhkan.
4. **Konfigurasi Docker (`Dockerfile`)** — instruksi build image berbasis `python:3.9-slim`.
5. **Setup Hugging Face Space** — New Space, SDK Docker, Visibility Public, buat Access Token (Write).
6. **Upload ke Cloud** — `git clone`, copy file, `git add`, `git commit`, `git push` (password = Access Token).
7. **Pengujian** — tunggu status *Building* → *Running*, lalu upload gambar untuk prediksi.

> Pada notebook ini file deployment ditulis ke folder `deployment/`. Server Flask **tidak dijalankan** (bersifat blocking); sebagai gantinya kita menguji logika `preprocess_image` secara lokal pada satu gambar test.

## Langkah 1-4: Membuat File Deployment (`app.py`, `requirements.txt`, `Dockerfile`)

Sel di bawah menuliskan ketiga file ke folder `deployment/`.

In [1]:
import os
os.makedirs('deployment', exist_ok=True)

app_py = r"""import os
import numpy as np
import cv2
import pickle
import tensorflow as tf
from flask import Flask, request, render_template_string
from skimage.feature import hog

app = Flask(__name__)

# Load Model & Scaler
MODEL_PATH = 'day_night_model.h5'
SCALER_PATH = 'scaler.pkl'

try:
    model = tf.keras.models.load_model(MODEL_PATH)
    with open(SCALER_PATH, 'rb') as f:
        scaler = pickle.load(f)
    print("✅ System Loaded Successfully")
except Exception as e:
    print(f"❌ Error loading system: {e}")

def preprocess_image(image_bytes):
    # Decode gambar
    nparr = np.frombuffer(image_bytes, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

    # Preprocessing (Harus sama persis dengan Training)
    img = cv2.resize(img, (256, 256))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    hog_feat = hog(gray, orientations=9, pixels_per_cell=(8,8),
                   cells_per_block=(2,2), block_norm='L2-Hys',
                   visualize=False, feature_vector=True)

    return scaler.transform(hog_feat.reshape(1, -1))

@app.route('/', methods=['GET'])
def home():
    return render_template_string('''
    <div style="text-align:center; padding:50px;">
        <h1>Day vs Night Classifier</h1>
        <form action="/predict" method="post" enctype="multipart/form-data">
            <input type="file" name="file" required><br><br>
            <button type="submit">Prediksi</button>
        </form>
    </div>
    ''')

@app.route('/predict', methods=['POST'])
def predict():
    try:
        file = request.files['file']
        data = preprocess_image(file.read())
        prediction = model.predict(data)[0][0]

        label = "Day (Siang)" if prediction > 0.5 else "Night (Malam)"
        return f"<h2 style='text-align:center'>Hasil: {label}</h2><center><a href='/'>Kembali</a></center>"
    except Exception as e:
        return f"Error: {e}"

if __name__ == '__main__':
    # Port 7860 wajib untuk Hugging Face Spaces
    app.run(host='0.0.0.0', port=7860)
"""

with open('deployment/app.py', 'w') as f:
    f.write(app_py)

requirements_txt = 'flask\ntensorflow-cpu\nnumpy\nscikit-learn\nscikit-image\nopencv-python-headless\ngunicorn\n'
with open('deployment/requirements.txt', 'w') as f:
    f.write(requirements_txt)

dockerfile = r"""# Base Image Python 3.9 Slim
FROM python:3.9-slim

# Install Library Sistem untuk OpenCV (Debian Bookworm/Trixie Compatible)
RUN apt-get update && apt-get install -y \
    libgl1 \
    libglib2.0-0 \
    && rm -rf /var/lib/apt/lists/*

# Setup User Non-Root (Standar Keamanan HF)
RUN useradd -m -u 1000 user
USER user
ENV PATH="/home/user/.local/bin:$PATH"

# Setup Direktori Kerja
WORKDIR /app

# Install Dependencies
COPY --chown=user ./requirements.txt requirements.txt
RUN pip install --no-cache-dir --upgrade -r requirements.txt

# Copy File Aplikasi
COPY --chown=user . /app

# Expose Port & Jalankan
EXPOSE 7860
CMD ["python", "app.py"]
"""

with open('deployment/Dockerfile', 'w') as f:
    f.write(dockerfile)

print('✅ File deployment dibuat:', os.listdir('deployment'))

✅ File deployment dibuat: ['requirements.txt', 'app.py', 'Dockerfile']


## Uji Inferensi Lokal

Memuat model + scaler hasil Praktikum 1 dan menjalankan logika `preprocess_image` yang sama pada satu gambar dari `../data/images/test/day/`.

In [2]:
import numpy as np
import cv2
import pickle
import glob
import tensorflow as tf
from skimage.feature import hog

model = tf.keras.models.load_model('day_night_model.h5')
with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

def preprocess_image(image_bytes):
    # Decode gambar
    nparr = np.frombuffer(image_bytes, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)

    # Preprocessing (Harus sama persis dengan Training)
    img = cv2.resize(img, (256, 256))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    hog_feat = hog(gray, orientations=9, pixels_per_cell=(8,8),
                   cells_per_block=(2,2), block_norm='L2-Hys',
                   visualize=False, feature_vector=True)

    return scaler.transform(hog_feat.reshape(1, -1))

sample_path = sorted(glob.glob('../data/images/test/day/*.jpg'))[0]
with open(sample_path, 'rb') as f:
    image_bytes = f.read()

data = preprocess_image(image_bytes)
prediction = model.predict(data, verbose=0)[0][0]
label = "Day (Siang)" if prediction > 0.5 else "Night (Malam)"

print("Sample:", sample_path)
print("Probabilitas:", prediction)
print("Hasil:", label)

I0000 00:00:1790788891.109736   34879 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790788891.110088   34879 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1790788891.148152   34879 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1790788892.415485   34879 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790788892.415735   34879 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


E0000 00:00:1790788893.825489   34879 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1790788893.825705   34946 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1790788893.841855   34879 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Sample: ../data/images/test/day/20151101_152050.jpg
Probabilitas: 1.0
Hasil: Day (Siang)
